In [ ]:
import pandas as pd

# load the full 2090 samples or the 1254 consensus samples
data_path = "./2090_ready_samples.csv"
# data_path = "./1254_ready_samples.csv'
with open(data_path, 'r') as f:
     data_ccc = pd.read_csv(f)

data_ccc

In [38]:
from openai import OpenAI
from together import Together
from tqdm import tqdm

# Create API client based on the specified API type
def create_client(API_type):

    if API_type == 'openai':
        key_file_path = './openai_api_key.txt'
        with open(key_file_path, 'r') as file:
            api_key = file.read().strip()
        client = OpenAI(api_key=api_key)

    elif API_type == 'together':
        key_file_path = './together_api_key.txt'
        with open(key_file_path, 'r') as file:
            api_key = file.read().strip()
        client = Together(api_key=api_key)

    return client

gpt_model = "gpt-4o-2024-08-06"
llama_model = "meta-llama/Llama-3-70B-chat-hf"

#samples
targets = data_ccc['target_compound'].tolist()
texts = data_ccc['text'].tolist()

#prompt variations
def create_Dutch_prompt_CandR(target, text):  # original Dutch with reason
    part_1 = "In deze annotatietaak vragen we u om te beoordelen of u de vetgedrukte term **"
    part_2 = """** omstreden vindt (contentious in het Engels) in de context van de tekst die eromheen staat, afgebakend door drievoudige aanhalingstekens. Voor het doel van deze taak beschouwen we een term als omstreden wanneer het volgens u mogelijk beledigend, denigrerend, kwetsend, of om een andere reden niet passend is naar huidige maatstaven. Om uw oordeel te vellen kunt u zich bijvoorbeeld afvragen of u de term in een soortgelijke zin zou gebruiken en/of u verrast zou zijn als de term op deze manier gebruikt zou worden door anderen. Als u een term in de gegeven zin een klein beetje omstreden vindt, antwoord dan 'omstreden naar huidige maatstaven'. Zo niet, antwoord dan 'niet omstreden'. En geef de reden.

\"\"\""""
    part_3 = """\"\"\""""
    return part_1 + target + part_2 + text + part_3


def create_Dutch_prompt_onlyC(target, text):  # original Dutch without reason
    part_1 = "In deze annotatietaak vragen we u om te beoordelen of u de vetgedrukte term **"
    part_2 = """** omstreden vindt (contentious in het Engels) in de context van de tekst die eromheen staat, afgebakend door drievoudige aanhalingstekens. Voor het doel van deze taak beschouwen we een term als omstreden wanneer het volgens u mogelijk beledigend, denigrerend, kwetsend, of om een andere reden niet passend is naar huidige maatstaven. Om uw oordeel te vellen kunt u zich bijvoorbeeld afvragen of u de term in een soortgelijke zin zou gebruiken en/of u verrast zou zijn als de term op deze manier gebruikt zou worden door anderen. Als u een term in de gegeven zin een klein beetje omstreden vindt, antwoord dan 'omstreden naar huidige maatstaven'. Zo niet, antwoord dan 'niet omstreden'. Geef geen reden.

\"\"\""""
    part_3 = """\"\"\""""
    return part_1 + target + part_2 + text + part_3


def create_simple_Dutch_prompt_CandR(target, text):  # simple Dutch with reason
    part_1 = """Naar huidige maatstaven, is het vetgedrukte woord **"""
    part_2 = """**, in de onderstaande context afgebakend door drievoudige aanhalingstekens, mogelijk beledigend, denigrerend of kwetsend? Antwoord met Ja of Nee en geef een reden.
    
\"\"\""""
    part_3 = """\"\"\""""
    return part_1 + target + part_2 + text + part_3


def create_simple_Ducth_prompt_onlyC(target, text):  # simple Dutch without reason
    part_1 = """Naar huidige maatstaven, is het vetgedrukte woord **"""
    part_2 = """**, in de onderstaande context afgebakend door drievoudige aanhalingstekens, mogelijk beledigend, denigrerend of kwetsend? Antwoord alleen met Ja of Nee.
    
\"\"\""""
    part_3 = """\"\"\""""
    return part_1 + target + part_2 + text + part_3


def create_English_prompt_CandR(target, text):  # original English with reason
    part_1 = "In this annotation task, we ask you to judge whether you find the bolded term **"
    part_2 = """** contentious in the context of the surrounding text, delimited by triple quotes. For the purposes of this task, we consider a term to be contentious when you think it is potentially offensive, derogatory, hurtful, or otherwise inappropriate by current standards. In making your judgment, you might ask yourself, for example, whether you would use the term in a similar sentence and/or whether you would be surprised if the term were used in this way by others. If you find a term in the given sentence slightly contentious, answer 'contentious according to current standards'. If not, answer 'not contentious'. And give a reason.
    
\"\"\""""
    part_3 = """\"\"\""""
    return part_1 + target + part_2 + text + part_3


def create_English_prompt_onlyC(target, text):  # original English without reason
    part_1 = "In this annotation task, we ask you to judge whether you find the bolded term **"
    part_2 = """** contentious in the context of the surrounding text, delimited by triple quotes. For the purposes of this task, we consider a term to be contentious when you think it is potentially offensive, derogatory, hurtful, or otherwise inappropriate by current standards. In making your judgment, you might ask yourself, for example, whether you would use the term in a similar sentence and/or whether you would be surprised if the term were used in this way by others. If you find a term in the given sentence slightly contentious, answer 'contentious according to current standards'. If not, answer 'not contentious'. Don't give a reason.
    
\"\"\""""
    part_3 = """\"\"\""""
    return part_1 + target + part_2 + text + part_3


def create_simple_English_prompt_CandR(target, text):  # simple English with reason
    part_1 = """Based on current standards, is the bolded word **"""
    part_2 = """** , in the following context delimited by triple quotes, potentially offensive, derogatory, or hurtful? Answer Yes or No and give a reason.
    
\"\"\""""
    part_3 = """\"\"\""""
    return part_1 + target + part_2 + text + part_3


def create_simple_English_prompt_onlyC(target, text):  # simple English without reason
    part_1 = """Based on current standards, is the bolded word **"""
    part_2 = """** , in the following context delimited by triple quotes, potentially offensive, derogatory, or hurtful? Answer only with Yes or No.
    
\"\"\""""
    part_3 = """\"\"\""""
    return part_1 + target + part_2 + text + part_3


def generate_response(client, targets, texts, prompt_type, model_name, temperature=0):
    model_output = []
    for i in tqdm(range(len(texts))):
        print(i)
        user_prompt = prompt_type(targets[i], texts[i])  
        # print(user_prompt)
        completion = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "user",
            "content": user_prompt}
        ],
        temperature=temperature
        )
        output = completion.choices[0].message.content
        model_output.append(output)
        print(output)
    return model_output

In [ ]:
# get the output of gpt-4o or llama-3-70b

model_output = generate_response(create_client("openai"), targets, texts, create_Dutch_prompt_onlyC, gpt_model, temperature=0)
# model_output = generate_response(create_client("together"), targets, texts, create_simple_prompt_onlyC,llama_model, temperature=0)

In [31]:
# convert the output to numbers (0 for not contentious, 1 for contentious)

def Dutch_label_to_number(label):
    if 'Niet omstreden' in label or 'niet omstreden' in label:
        return 0
    elif 'omstreden' in label or 'Omstreden' in label:
        return 1
    else:
        return label
    

def simple_Dutch_label_to_number(label):
    if 'Nee' in label or 'nee' in label:
        return 0
    elif 'Ja' in label or 'ja' in label:
        return 1
    else:
        return label
    

def English_label_to_number(label):
    if 'Not contentious' in label or 'not contentious' in label:
        return 0
    elif 'Contentious' in label or 'contentious' in label:
        return 1
    else:
        return label
    

def simple_English_label_to_number(label):
    if 'No' in label or 'no' in label:
        return 0
    elif 'Yes' in label or 'yes' in label:
        return 1
    else:
        return label
    
    
# choose the right conversion function based on the prompt type
converted_output = [Dutch_label_to_number(content) for content in model_output]

# make a new dataset including the new results
updated_data = data_ccc.copy()
updated_data['output_score'] = converted_output
updated_data['output'] = model_output

# check if any output_score is not 0 or 1
updated_data[updated_data['output_score'] != 0][updated_data['output_score'] != 1]

updated_data

/var/folders/5w/511b227d2z53cxgmdw8rgp400000gp/T/ipykernel_52758/2315311626.py:46: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  updated_data[updated_data['output_score'] != 0][updated_data['output_score'] != 1]


,extract_id,target,target_compound,text,score,output_score,output
0,10,roots,roots,"50 A Westerhof 2nd 25 Pk Early Ohio, Avd Veere...",0.0,0,niet omstreden
1,1000,moslim,Moslims,"I Zijn optreden, na zijn terugkeer op Trinidad...",0.0,0,niet omstreden
2,1002,hottentot,Hottentot,De meeste mannen zyn jonge kerels tusschen 30 ...,1.0,1,omstreden naar huidige maatstaven
3,1003,moslim,Moslim,"De Turksche grondwet, welker inhoud!\n\nwij re...",0.0,0,niet omstreden
4,1005,aboriginal,Aboriginal,Dit leidde in Engeland tot het optreden der Ab...,0.0,0,Niet omstreden.
...,...,...,...,...,...,...,...
2085,c0,kaffer,Kaffer,Deze laatste der serie kennisvoorstellingen in...,1.0,1,omstreden naar huidige maatstaven
2086,c1,neger,negerin,"ao niet vergeten zijn, en vertegenwoordigd wor...",1.0,1,omstreden naar huidige maatstaven
2087,c2,achtergrond,achtergrond,Maar wees niet te spoedig ontroostbaar.\n\nWij...,0.0,0,Niet omstreden.
2088,c3,dwerg,dwerg,De trommen roffelden.\n\nVóór de afreis gingen...,1.0,1,omstreden naar huidige maatstaven


In [ ]:
# Save the updated dataset to a new CSV file in a separate folder for future analysis.
# change the name of the file based on the model and prompt type.
# options are: [gpt-4o or llama-3-70b] + [ori or simp]_[nl or en]_[C or CandR]

# Here is an example:
sav_file = './gpt and llama/gpt-4o ori_nl_C.csv' 
with open(sav_file, 'w') as f:
    updated_data.to_csv(f)